# ARC AGI 3 — Warm-start submission (v1)

Replay-warm-start agent: plays GT replay actions verbatim on public games (where we have the replay), random fallback on hidden games. Honors RESET markers (`action_input.id == 0` while state is `GAME_OVER`).

Local smoke test on sp80, ar25, lp85: replay-warm-start lifts levels from 0/0/0 (random) to **3 / 4 / 6** in 192 steps. With `MAX_ACTIONS=inf` (gateway-controlled budget, ≤ 12h total) the same replays should reach WIN if the gateway lets us run them in full (468/577/419 actions to full WIN).

Patterns adopted from `arc3-sample-submission-stochastic-goose.ipynb`:
- `MAX_ACTIONS = float('inf')` — let gateway cap the per-game budget.
- `_MAX_FRAMES = 10` sliding window on `self.frames` to bound memory.
- Try/except wrappers around `choose_action` and `is_done` with random fallback.
- Time-elapsed safety net for the 12h notebook cap.
- DEBUG print on the first action call to introspect gateway frame format.

**Two pieces of input data needed:**
1. `/kaggle/input/competitions/arc-prize-2026-arc-agi-3/` — comes free with the competition (wheels + framework).
2. **Your own Kaggle Dataset** with the GT replays. Expected layout:
   ```
   /kaggle/input/<your-dataset-slug>/environment_files/<game_id>/replays/*.json
   ```
   Update `REPLAY_INPUT` in cell 2 to your dataset slug.

See `kaggle_notebook/SUBMIT.md` for full CLI submission steps.

In [1]:
# --- Cell 1: install vendored wheels --- #
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatib

In [2]:
# --- Cell 2: configure paths --- #
from pathlib import Path

# 1. Competition input — provided by Kaggle, contains framework + wheels.
COMPETITION_INPUT = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')

# 2. YOUR dataset — must contain the 25 games' replay JSONs at:
#       <REPLAY_INPUT>/environment_files/<short_game_id>/replays/*.json
#    Update the slug to match your uploaded dataset.
REPLAY_INPUT = Path('/kaggle/input/arc-agi-3-replays-v1')

# Working dir Kaggle exposes as writable.
WORK = Path('/kaggle/working')

# Where the agent will look up replays at runtime.
REPLAY_BASE_DIR = WORK / 'replays'
REPLAY_BASE_DIR.mkdir(exist_ok=True)

assert COMPETITION_INPUT.exists(), f'Competition input missing: {COMPETITION_INPUT}'
if not REPLAY_INPUT.exists():
    print(f'WARNING: REPLAY_INPUT not found at {REPLAY_INPUT}. Agent will run fully random.')
print('paths OK')

paths OK


In [3]:
# --- Cell 3: stage replay files into REPLAY_BASE_DIR --- #
import os

if REPLAY_INPUT.exists():
    src_root = REPLAY_INPUT / 'environment_files'
    if not src_root.is_dir():
        src_root = REPLAY_INPUT  # tolerate flat layout
    n_games_with_replays = 0
    for game_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        replays_src = game_dir / 'replays'
        if not replays_src.is_dir():
            continue
        replays_dst = REPLAY_BASE_DIR / game_dir.name / 'replays'
        replays_dst.parent.mkdir(parents=True, exist_ok=True)
        if replays_dst.exists() or replays_dst.is_symlink():
            continue
        os.symlink(replays_src, replays_dst)
        n_games_with_replays += 1
    print(f'staged replays for {n_games_with_replays} games into {REPLAY_BASE_DIR}')
else:
    print('no REPLAY_INPUT — agent will run with no replays (random fallback)')

os.environ['ARC_REPLAY_BASE_DIR'] = str(REPLAY_BASE_DIR)

no REPLAY_INPUT — agent will run with no replays (random fallback)


In [4]:
%%writefile /kaggle/working/my_agent.py
# =====================================================================
# WarmstartAgent — replay-warm-start v1 for ARC AGI 3.
#
# Plays GT replay actions verbatim until exhausted; random fallback after.
# Honors RESET markers (action_input.id == 0 while state == GAME_OVER) so
# the human's recovery path through deaths is reproduced.
#
# Patterns from arc3-sample-submission-stochastic-goose.ipynb:
#   - MAX_ACTIONS = inf (gateway caps).
#   - _MAX_FRAMES = 10 sliding window via append_frame override.
#   - try/except around is_done + choose_action with random fallback.
#   - 12h - 5min wall-clock safety net.
#   - DEBUG print on first action to introspect gateway frame format.
#   - available_actions iterable supports both ints and GameAction enums.
# =====================================================================
from __future__ import annotations

import json
import os
import random
import time
import traceback
from pathlib import Path
from typing import Any, Dict, List, Optional

from arcengine import FrameData, GameAction, GameState

from agents.agent import Agent


REPLAY_BASE_DIR = Path(os.environ.get('ARC_REPLAY_BASE_DIR', '/kaggle/working/replays'))
WALL_BUDGET_SECONDS = 12 * 3600 - 5 * 60  # 12h - 5min safety margin


def _normalize_action_id(raw: Any) -> int:
    if raw is None:
        return 0
    if isinstance(raw, bool):
        return 0
    if isinstance(raw, int):
        return raw if 1 <= raw <= 7 else 0
    if isinstance(raw, str):
        s = raw.strip().upper()
        if s == 'RESET':
            return 0
        if s.startswith('ACTION') and s[6:].isdigit():
            n = int(s[6:])
            return n if 1 <= n <= 7 else 0
    return 0


def _load_replay_actions(short_game_id: str) -> List[Dict[str, Any]]:
    """Returns a list of {'type': 'reset' | 'action', ...} entries."""
    replay_dir = REPLAY_BASE_DIR / short_game_id / 'replays'
    if not replay_dir.is_dir():
        return []
    files = sorted(replay_dir.glob('*.json'))
    if not files:
        return []
    out: List[Dict[str, Any]] = []
    try:
        with open(files[0], 'r') as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                ai = (rec.get('data') or {}).get('action_input') or {}
                aid = _normalize_action_id(ai.get('id'))
                if aid == 0:
                    out.append({'type': 'reset'})
                    continue
                ad = ai.get('data') or {}
                try:
                    x = int(ad.get('x', 0)) if aid == 6 else 0
                    y = int(ad.get('y', 0)) if aid == 6 else 0
                except (TypeError, ValueError):
                    x, y = 0, 0
                out.append({'type': 'action', 'id': aid, 'x': x, 'y': y})
    except Exception:
        return []
    return out


def _available_action_ids(latest_frame: FrameData) -> List[int]:
    """Gateway sends ints [1..6]; toolkit sends GameAction enums. Handle both."""
    raw = getattr(latest_frame, 'available_actions', None)
    if raw is None:
        return list(range(1, 8))
    out = []
    for a in raw:
        try:
            v = a.value if hasattr(a, 'value') else int(a)
            if 1 <= int(v) <= 7:
                out.append(int(v))
        except Exception:
            continue
    return out or list(range(1, 8))


def _safe_random_action(rng: random.Random, latest_frame: FrameData) -> GameAction:
    """Random non-RESET action, restricted to available_actions when known."""
    available_ids = _available_action_ids(latest_frame)
    aid = rng.choice(available_ids) if available_ids else 1
    try:
        action = GameAction.from_id(int(aid))
    except Exception:
        action = GameAction.ACTION1
    if action.is_complex():
        action.set_data({'x': rng.randint(0, 63), 'y': rng.randint(0, 63)})
        action.reasoning = {'phase': 'fallback', 'policy': 'random_action6'}
    elif action.is_simple():
        action.reasoning = 'fallback random'
    return action


class MyAgent(Agent):
    """Replay-warm-start agent."""

    MAX_ACTIONS = float('inf')
    _MAX_FRAMES = 10

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self._short_id = self.game_id.split('-', 1)[0] if self.game_id else ''
        self._replay = _load_replay_actions(self._short_id)
        self._replay_idx = 0
        self._start_time = time.time()
        self._debug_logged = False
        seed_int = (int(time.time() * 1_000_000) + hash(self.game_id)) & 0xFFFFFFFF
        self._rng = random.Random(seed_int)
        print(
            f'[MyAgent] init game_id={self.game_id} short_id={self._short_id} '
            f'replay_entries={len(self._replay)}',
            flush=True,
        )

    def append_frame(self, frame: FrameData) -> None:
        """Sliding window on self.frames to bound memory across long replays."""
        self.frames.append(frame)
        if len(self.frames) > self._MAX_FRAMES:
            self.frames = self.frames[-self._MAX_FRAMES:]
        if frame.guid:
            self.guid = frame.guid
        if hasattr(self, 'recorder') and not self.is_playback:
            try:
                self.recorder.record(json.loads(frame.model_dump_json()))
            except Exception:
                pass

    def _wall_elapsed(self) -> bool:
        return (time.time() - self._start_time) >= WALL_BUDGET_SECONDS

    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:
        try:
            if latest_frame.state is GameState.WIN:
                return True
            if self._wall_elapsed():
                print(f'[MyAgent] {self._short_id} wall budget elapsed', flush=True)
                return True
            return False
        except Exception as exc:
            print(f'[MyAgent] is_done crashed: {exc}', flush=True)
            traceback.print_exc()
            return True  # bail safely on error

    def choose_action(
        self, frames: List[FrameData], latest_frame: FrameData
    ) -> GameAction:
        try:
            # First-action introspection — useful for debugging gateway format
            if not self._debug_logged:
                self._debug_logged = True
                print(
                    f'[MyAgent] first action for {self._short_id} '
                    f'state={latest_frame.state} levels={getattr(latest_frame, "levels_completed", "?")} '
                    f'available_actions={getattr(latest_frame, "available_actions", "?")}',
                    flush=True,
                )

            # Phase A: replay-warm-start
            while self._replay_idx < len(self._replay):
                entry = self._replay[self._replay_idx]
                self._replay_idx += 1
                if entry['type'] == 'reset':
                    if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                        action = GameAction.RESET
                        action.reasoning = {'phase': 'warmstart', 'marker': 'reset'}
                        return action
                    continue
                aid = int(entry['id'])
                try:
                    action = GameAction.from_id(aid)
                except Exception:
                    continue
                if action.is_complex():
                    action.set_data({'x': int(entry['x']), 'y': int(entry['y'])})
                    action.reasoning = {
                        'phase': 'warmstart',
                        'replay_idx': self._replay_idx,
                    }
                elif action.is_simple():
                    action.reasoning = f'warmstart replay_idx={self._replay_idx}'
                return action

            # Phase B: replay exhausted or hidden game
            if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                action = GameAction.RESET
                action.reasoning = {'phase': 'fallback', 'marker': 'reset'}
                return action
            return _safe_random_action(self._rng, latest_frame)

        except Exception as exc:
            print(
                f'[MyAgent] choose_action crashed at counter={self.action_counter}: '
                f'{type(exc).__name__}: {exc}',
                flush=True,
            )
            traceback.print_exc()
            return _safe_random_action(self._rng, latest_frame)

Writing /kaggle/working/my_agent.py


In [5]:
# --- Cell 5: in rerun mode, set up the framework and run main.py --- #
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway HTTP service to be ready
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the framework to writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Drop our agent into the framework's templates folder
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Replace agents/__init__.py with a minimal one that only imports our
    # agent + Random. The original eagerly imports llm / langgraph templates
    # whose deps aren't installed.
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    # Write a .env that points the framework at the gateway.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
ARC_REPLAY_BASE_DIR=/kaggle/working/replays
""")

    # Run the agent. main.py iterates the gateway's games and invokes MyAgent.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        ARC_REPLAY_BASE_DIR=/kaggle/working/replays \
        python main.py --agent myagent

In [6]:
# --- Cell 6: in dev mode, write dummy submission.parquet --- #
# The grader replaces this with real scoring during rerun.
import os

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('dummy submission.parquet written (dev mode)')

dummy submission.parquet written (dev mode)
